email agent
- authenticates user
    - only then are they allowed into the "inbox"
    - dynamic tools and prompt on the condition of there being an email and password in state that match hardcoded
- checks "inbox"
    - email in tool
- sends emails
    - human in the loop

In [1]:
import warnings

warnings.filterwarnings("ignore", message=".*Pydantic serializer warnings.*", category=UserWarning)

from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from dataclasses import dataclass


@dataclass
class EmailContext:
    email_address: str = "julie@example.com"
    password: str = "password123"

In [3]:
from langchain.agents import AgentState


class AuthenticatedState(AgentState):
    authenticated: bool

In [4]:
from langchain.messages import ToolMessage
from langchain.tools import ToolRuntime, tool
from langgraph.types import Command


@tool
def check_inbox() -> str:
    """Check the inbox for recent emails"""
    return """
    Hi Julie, 
    I'm going to be in town next week and was wondering if we could grab a coffee?
    - best, Jane (jane@example.com)
    """

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an response email"""
    return f"Email sent to {to} with subject {subject} and body {body}"

@tool
def authenticate(email: str, password: str, runtime: ToolRuntime) -> Command:
    """Authenticate the user with the given email and password"""
    if email == runtime.context.email_address and password == runtime.context.password:
        return Command(update={
            "authenticated": True, 
            "messages": [ToolMessage(
                "Successfully authenticated", 
                tool_call_id=runtime.tool_call_id)]
        })
    else:
        return Command(update={
            "authenticated": False,
            "messages": [ToolMessage(
                "Authentication failed", 
                tool_call_id=runtime.tool_call_id)]
        })

In [5]:
from _collections_abc import Callable

from langchain.agents.middleware import ModelRequest, ModelResponse, wrap_model_call


@wrap_model_call
def dynamic_tool_call(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:

    """Allow read inbox and send email tools only if user provides correct email and password"""

    authenticated = request.state.get("authenticated")
    
    if authenticated:
        tools = [check_inbox, send_email]
    else:
        tools = [authenticate]

    request = request.override(tools=tools) 
    return handler(request)

>Note: the prompts were modified since filming to constrain the model to more reliably match the filmed sequence. You may still experience different responses from the model, which is expected. You may need to modify the human message to provide appropriate responses.

In [6]:
from langchain.agents.middleware import dynamic_prompt

authenticated_prompt = """You are a helpful assistant that can check the inbox and send emails. 
Your first step after authentication is to check the inbox."""
unauthenticated_prompt = "You are a helpful assistant that can authenticate users."

@dynamic_prompt
def dynamic_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on authentication status"""
    authenticated = request.state.get("authenticated")

    if authenticated:
        return authenticated_prompt
    else:
        return unauthenticated_prompt

In [7]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    "gpt-5-nano",
    tools=[authenticate, check_inbox, send_email],
    checkpointer=InMemorySaver(),
    state_schema=AuthenticatedState,
    context_schema=EmailContext,
    middleware=[
        dynamic_tool_call, 
        dynamic_prompt,
        HumanInTheLoopMiddleware(
            interrupt_on={
                "authenticate": False,
                "check_inbox": False,
                "send_email": True,
            })
        ]
    )

In [8]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="julie@example.com, password123")]},
    context=EmailContext(),
    config=config
)

print(response['messages'][-1].content)

I found Jane’s message: “Hi Julie, I’m going to be in town next week and was wondering if we could grab a coffee?”

Would you like me to reply? I can send a draft for you or customize it. Here are a few options:

Option 1 — friendly and flexible
Hi Jane, that sounds great! I’d love to catch up. What day next week works for you? I’m generally free Mon/Wed mornings and Tue/Thu afternoons. If you have a preferred time or coffee spot, let me know and we’ll meet there.

Option 2 — suggest specific times
Hi Jane, that would be lovely. How about Tuesday at 10:30am or Thursday at 4:00pm? If those don’t work, feel free to suggest another time. Looking forward to it!

Option 3 — ultra-brief
Hi Jane, I’d love to! What day/time works for you next week?

Notes you can add if you want:
- Include a location suggestion (e.g., “at our usual cafe on Main Street” or “anywhere you prefer”)
- Ask about the coffee shop preference
- Offer to send a calendar invite

Tell me which option you prefer, or provide

In [9]:

response = agent.invoke(
    {"messages": [HumanMessage(content="any draft is fine. don't check back.")]},
    context=EmailContext(),
    config=config
)

print(response['messages'][-1].tool_calls)

[{'name': 'send_email', 'args': {'to': 'jane@example.com', 'subject': 'Re: Coffee next week', 'body': 'Hi Jane, that sounds great! I’d love to catch up. What day next week works for you? I’m generally free Mon/Wed mornings and Tue/Thu afternoons. If you have a preferred time or coffee spot, let me know and we’ll meet there. Looking forward to it!'}, 'id': 'call_wAXkXQF1qhlGj8yqiFFUp66l', 'type': 'tool_call'}]


In [10]:
response

{'messages': [HumanMessage(content='julie@example.com, password123', additional_kwargs={}, response_metadata={}, id='b79aabdc-934d-4f30-9633-eaa524c916a7'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 94, 'prompt_tokens': 151, 'total_tokens': 245, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EAej1V4dhlfW2ByCjmUuwBj9WPRLu', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fe253-3f48-74e2-a905-8b43ae22ee44-0', tool_calls=[{'name': 'authenticate', 'args': {'email': 'julie@example.com', 'password': 'password123'}, 'id': 'call_N4b5UoBHIruUV9mStW4y2i8A', 'type': 'tool_call'}], invalid_tool_calls=[], usage_meta

In [11]:
for message in response['messages']:
    if message:
        print(type(message))
        for field, value in message:
            print(f"{field}={value!r}")
        print()

<class 'langchain_core.messages.human.HumanMessage'>
content='julie@example.com, password123'
additional_kwargs={}
response_metadata={}
type='human'
name=None
id='1b5565fa-072c-4e88-a880-7811516ce90d'

<class 'langchain_core.messages.ai.AIMessage'>
content=''
additional_kwargs={'refusal': None}
response_metadata={'token_usage': {'completion_tokens': 158, 'prompt_tokens': 151, 'total_tokens': 309, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EAeeCKrFdI28PDax5NEwfN34cuQRC', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}
type='ai'
name=None
id='lc_run--019fe24e-af0a-71f3-86c1-cbf56715704d-0'
tool_calls=[{'name': 'authenticate', 'args': {'email': 'julie@example.com', 'password': 'password123'

In [11]:
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Hi Jane, that sounds great! I’d love to catch up. What day next week works for you? I’m generally free Mon/Wed mornings and Tue/Thu afternoons. If you have a preferred time or coffee spot, let me know and we’ll meet there. Looking forward to it!


In [12]:
from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}  # or "reject"
    ), 
    config=config # Same thread ID to resume the paused conversation
)

print(response["messages"][-1].content)

Email sent to jane@example.com with subject Re: Coffee next week.

Body:
Hi Jane, that sounds great! I’d love to catch up. What day next week works for you? I’m generally free Mon/Wed mornings and Tue/Thu afternoons. If you have a preferred time or coffee spot, let me know and we’ll meet there. Looking forward to it!

I won’t check for replies unless you ask me to. If you’d like, I can draft a follow-up or set a reminder when she responds.


In [13]:
from pprint import pprint

pprint(response)

{'authenticated': True,
 'messages': [HumanMessage(content='julie@example.com, password123', additional_kwargs={}, response_metadata={}, id='b79aabdc-934d-4f30-9633-eaa524c916a7'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 94, 'prompt_tokens': 151, 'total_tokens': 245, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EAej1V4dhlfW2ByCjmUuwBj9WPRLu', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fe253-3f48-74e2-a905-8b43ae22ee44-0', tool_calls=[{'name': 'authenticate', 'args': {'email': 'julie@example.com', 'password': 'password123'}, 'id': 'call_N4b5UoBHIruUV9mStW4y2i8A', 'type': 'tool_call'}